# LocallyConnected2D (Non-shared Parameters)

In [ ]:
# import kagglehub

# path = kagglehub.dataset_download("puneet6060/intel-image-classification")
# print("Path to dataset files:", path)

## Import Libraries

In [11]:
import os
import sys
import json
import time
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from sklearn.metrics import f1_score, classification_report

sys.path.append('../../../')

from src.cnn.layers.locally_connected import LocallyConnected2DLayer
from src.cnn.layers.pooling import MaxPooling2DLayer, AveragePooling2DLayer
from src.cnn.layers.flatten import FlattenLayer
from src.cnn.layers.conv2d import Conv2DLayer
from src.shared.layer import Dense
from src.shared.activations import Activation
from src.cnn.model import CNNModel

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TF version:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices('GPU'))
tf.test.is_built_with_cuda()

TF version: 2.21.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


True

## Config

In [12]:
TRAIN_DIR = '../../data/intel/seg_train'
TEST_DIR  = '../../data/intel/seg_test'
MODEL_DIR  = '../../data/models/cnn'
os.makedirs(MODEL_DIR, exist_ok=True)

IMG_SIZE    = (150, 150)
BATCH_SIZE  = 32
EPOCHS      = 10
NUM_CLASSES = 6
LR          = 1e-4
VAL_SPLIT   = 0.2
MODEL_NAME  = 'locally_connected'

## Import Dataset

In [13]:
AUTOTUNE      = tf.data.AUTOTUNE
normalization = keras.layers.Rescaling(1.0 / 255.0)

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=True,
)
class_names = train_ds.class_names

val_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR,
    validation_split=VAL_SPLIT,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False,
)

train_ds = (
    train_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .shuffle(buffer_size=1000, seed=SEED, reshuffle_each_iteration=True)
    .prefetch(buffer_size=AUTOTUNE)
)

val_ds = (
    val_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .cache()
    .prefetch(buffer_size=AUTOTUNE)
)

test_ds = (
    test_ds
    .map(lambda x, y: (normalization(x), y), num_parallel_calls=AUTOTUNE)
    .prefetch(buffer_size=AUTOTUNE)
)

train_count = sum(1 for _ in tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="training",
    seed=SEED, image_size=IMG_SIZE, batch_size=1, label_mode="int"
))
val_count = sum(1 for _ in tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, validation_split=VAL_SPLIT, subset="validation",
    seed=SEED, image_size=IMG_SIZE, batch_size=1, label_mode="int"
))
test_count = sum(1 for _ in tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=1, label_mode="int", shuffle=False
))

print("Kelas      :", {name: i for i, name in enumerate(class_names)})
print(f"Train      : {train_count} gambar")
print(f"Validation : {val_count} gambar")
print(f"Test       : {test_count} gambar")

Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Found 3000 files belonging to 6 classes.
Found 14034 files belonging to 6 classes.
Using 11228 files for training.
Found 14034 files belonging to 6 classes.
Using 2806 files for validation.
Found 3000 files belonging to 6 classes.
Kelas      : {'buildings': 0, 'forest': 1, 'glacier': 2, 'mountain': 3, 'sea': 4, 'street': 5}
Train      : 11228 gambar
Validation : 2806 gambar
Test       : 3000 gambar


## Model Definition

In [14]:
# Load weights dari model Conv2D
conv2d_model = keras.models.load_model('Layer-2-32-64-5x5-maxpool.h5')

print(">>> Loaded Keras Model Summary")
conv2d_model.summary()

print("\n>>> Dense Layer Weights")
for layer in conv2d_model.layers:
    if isinstance(layer, Dense):
        w = layer.get_weights()[0]
        print(f"{layer.name}: input={w.shape[0]}, output={w.shape[1]}")


>>> Loaded Keras Model Summary


Model: "Layer-2-32-64-5x5-maxpool"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2s_1 (Conv2D)               │ (None, 146, 146, 32)   │         2,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_1 (MaxPooling2D)        │ (None, 73, 73, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2s_2 (Conv2D)               │ (None, 69, 69, 64)     │        51,264 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ maxpool_2 (MaxPooling2D)        │ (None, 34, 34, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 73984)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 128)            │     9,470,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ output (Dense)                  │ (None, 6)              │           774 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 9,524,552 (36.33 MB)

 Trainable params: 9,524,550 (36.33 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2 (12.00 B)


>>> Dense Layer Weights


In [15]:
print("\n>>> Keras Model Architecture")
for i, layer in enumerate(conv2d_model.layers):
    layer_type = type(layer).__name__
    print(f"\n[{i}] {layer.name} ({layer_type})")
    
    cfg = layer.get_config()
    
    if layer_type == 'Conv2D':
        print(f"    Kernel size: {cfg['kernel_size']}, Strides: {cfg['strides']}, Filters: {layer.filters}")
        if layer.get_weights():
            w = layer.get_weights()[0]
            print(f"    Weights shape: {w.shape}")
            
    elif layer_type in ('MaxPooling2D', 'AveragePooling2D'):
        print(f"    Pool size: {cfg['pool_size']}, Strides: {cfg['strides']}")
        
    elif layer_type == 'Dense':
        if layer.get_weights():
            w = layer.get_weights()[0]
            print(f"    Input: {w.shape[0]}, Output: {w.shape[1]}")
            
    elif layer_type == 'Flatten':
        print(f"    Flattens to: {cfg.get('data_format', 'channels_last')}")


>>> Keras Model Architecture

[0] conv2s_1 (Conv2D)
    Kernel size: (5, 5), Strides: (1, 1), Filters: 32
    Weights shape: (5, 5, 3, 32)

[1] maxpool_1 (MaxPooling2D)
    Pool size: (2, 2), Strides: (2, 2)

[2] conv2s_2 (Conv2D)
    Kernel size: (5, 5), Strides: (1, 1), Filters: 64
    Weights shape: (5, 5, 32, 64)

[3] maxpool_2 (MaxPooling2D)
    Pool size: (2, 2), Strides: (2, 2)

[4] flatten (Flatten)
    Flattens to: channels_last

[5] dense_1 (Dense)
    Input: 73984, Output: 128

[6] output (Dense)
    Input: 128, Output: 6


In [16]:
def get_one_test_image(test_ds):
    X_batch, _ = next(iter(test_ds))
    return X_batch[0].numpy()


def conv_valid_out_size(in_size, kernel_size, stride):
    return (in_size - kernel_size) // stride + 1


test_image = get_one_test_image(test_ds)
current_shape = test_image.shape

print(">>> Shape Propagation Conv2D Model")
print(f"Input: {current_shape}")

for i, layer in enumerate(conv2d_model.layers):
    if isinstance(layer, keras.layers.InputLayer):
        continue

    cfg = layer.get_config()
    layer_type = type(layer).__name__

    if layer_type == "Conv2D":
        kH, kW = cfg["kernel_size"]
        sH, sW = cfg["strides"]

        H, W, C = current_shape

        out_H = conv_valid_out_size(H, kH, sH)
        out_W = conv_valid_out_size(W, kW, sW)
        out_C = cfg["filters"]

        next_shape = (out_H, out_W, out_C)

        print(
            f"[{i}] Conv2D "
            f"(kernel={cfg['kernel_size']}, stride={cfg['strides']}): "
            f"{current_shape} -> {next_shape}"
        )

        current_shape = next_shape

    elif layer_type in ("MaxPooling2D", "AveragePooling2D"):
        pH, pW = cfg["pool_size"]
        sH, sW = cfg["strides"]

        H, W, C = current_shape

        out_H = conv_valid_out_size(H, pH, sH)
        out_W = conv_valid_out_size(W, pW, sW)

        next_shape = (out_H, out_W, C)

        print(
            f"[{i}] {layer_type} "
            f"(pool={cfg['pool_size']}, stride={cfg['strides']}): "
            f"{current_shape} -> {next_shape}"
        )

        current_shape = next_shape

    elif layer_type == "Flatten":
        H, W, C = current_shape
        flat_size = H * W * C

        next_shape = (flat_size,)

        print(
            f"[{i}] Flatten: "
            f"{current_shape} -> {next_shape}"
        )

        current_shape = next_shape

    elif layer_type == "Dense":
        weights = layer.get_weights()[0]
        keras_in_features, keras_out_features = weights.shape

        print(
            f"[{i}] Dense: "
            f"scratch/current input={current_shape[0]}, "
            f"keras expected input={keras_in_features}, "
            f"output={keras_out_features}"
        )

        current_shape = (keras_out_features,)

print(f"\nFinal simulated output shape: {current_shape}")

>>> Shape Propagation Conv2D Model
Input: (150, 150, 3)
[0] Conv2D (kernel=(5, 5), stride=(1, 1)): (150, 150, 3) -> (146, 146, 32)
[1] MaxPooling2D (pool=(2, 2), stride=(2, 2)): (146, 146, 32) -> (73, 73, 32)
[2] Conv2D (kernel=(5, 5), stride=(1, 1)): (73, 73, 32) -> (69, 69, 64)
[3] MaxPooling2D (pool=(2, 2), stride=(2, 2)): (69, 69, 64) -> (34, 34, 64)
[4] Flatten: (34, 34, 64) -> (73984,)
[5] Dense: scratch/current input=73984, keras expected input=73984, output=128
[6] Dense: scratch/current input=128, keras expected input=128, output=6

Final simulated output shape: (6,)


## Cek Shape Bobot LC2D

In [17]:
scratch_layers = []
correct_flatten_size = None

def calc_out_shape_hw(in_hw, kernel_size, stride):
    H, W = in_hw
    kH, kW = kernel_size
    return (H - kH) // stride + 1, (W - kW) // stride + 1

if len(IMG_SIZE) == 2:
    current_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
else:
    current_shape = IMG_SIZE

for layer in conv2d_model.layers:
    layer_type = type(layer).__name__

    if layer_type == 'InputLayer':
        continue

    if layer_type == 'Conv2D':
        conv_w, conv_b = layer.get_weights()
        cfg = layer.get_config()

        kH, kW, Cin_w, Cout = conv_w.shape
        stride = cfg['strides'][0]

        H, W, Cin_shape = current_shape

        if Cin_shape != Cin_w:
            raise ValueError(
                f"Channel mismatch di {layer.name}: "
                f"current_shape punya Cin={Cin_shape}, tetapi weight Conv2D punya Cin={Cin_w}"
            )

        out_H, out_W = calc_out_shape_hw((H, W), (kH, kW), stride)
        out_positions = out_H * out_W
        flat_conv_w = conv_w.reshape((kH * kW * Cin_w, Cout))

        lc_w = np.tile(flat_conv_w, (out_positions, 1, 1))
        lc_b = np.tile(conv_b, (out_positions, 1))

        scratch_layers.append(
            LocallyConnected2DLayer(weights=[lc_w, lc_b], config=cfg)
        )

        print(
            f"  {layer.name} (Conv2D) -> LocallyConnected2DLayer: "
            f"{conv_w.shape} -> {lc_w.shape}, output shape: "
            f"{current_shape} -> {(out_H, out_W, Cout)}"
        )

        current_shape = (out_H, out_W, Cout)

    elif layer_type == 'MaxPooling2D':
        cfg = layer.get_config()
        pool_size = cfg['pool_size']
        stride = cfg['strides'][0]

        H, W, C = current_shape
        out_H, out_W = calc_out_shape_hw((H, W), pool_size, stride)

        scratch_layers.append(MaxPooling2DLayer(layer))

        print(
            f"  {layer.name} (MaxPooling2D) -> MaxPooling2DLayer: "
            f"{current_shape} -> {(out_H, out_W, C)}"
        )

        current_shape = (out_H, out_W, C)

    elif layer_type == 'Flatten':
        scratch_layers.append(FlattenLayer())

        H, W, C = current_shape
        correct_flatten_size = H * W * C

        print(
            f"  {layer.name} (Flatten) -> FlattenLayer: "
            f"{current_shape} -> ({correct_flatten_size},)"
        )

        current_shape = (correct_flatten_size,)

    elif layer_type == 'Dense':
        keras_weights, keras_biases = layer.get_weights()
        keras_in_features, out_features = keras_weights.shape

        in_features = current_shape[0]

        dense_layer = Dense(in_features, out_features)

        if keras_in_features == in_features:
            dense_layer.set_weights(keras_weights, keras_biases)
            weight_status = "transfer Keras weights"
        else:
            weight_status = (
                f"random/zero init, skip Keras weights "
                f"(Keras input={keras_in_features}, scratch input={in_features})"
            )

        scratch_layers.append(dense_layer)

        print(
            f"  {layer.name} (Dense) -> DenseLayer({in_features}, {out_features}) "
            f"[{weight_status}]"
        )

        activation_name = layer.get_config().get('activation', 'linear')
        if activation_name != 'linear':
            from src.cnn.model import ActivationLayer
            scratch_layers.append(ActivationLayer(activation_name))
            print(f"  + ActivationLayer({activation_name})")

        current_shape = (out_features,)

model_scratch = CNNModel.from_layers(scratch_layers)

print("\n>>> Transfer bobot selesai!")
print(f">>> Final output shape tracking: {current_shape}")

  conv2s_1 (Conv2D) -> LocallyConnected2DLayer: (5, 5, 3, 32) -> (21316, 75, 32), output shape: (150, 150, 3) -> (146, 146, 32)
  maxpool_1 (MaxPooling2D) -> MaxPooling2DLayer: (146, 146, 32) -> (73, 73, 32)
  conv2s_2 (Conv2D) -> LocallyConnected2DLayer: (5, 5, 32, 64) -> (4761, 800, 64), output shape: (73, 73, 32) -> (69, 69, 64)
  maxpool_2 (MaxPooling2D) -> MaxPooling2DLayer: (69, 69, 64) -> (34, 34, 64)
  flatten (Flatten) -> FlattenLayer: (34, 34, 64) -> (73984,)
  dense_1 (Dense) -> DenseLayer(73984, 128) [transfer Keras weights]
  + ActivationLayer(relu)
  output (Dense) -> DenseLayer(128, 6) [transfer Keras weights]
  + ActivationLayer(softmax)

>>> Transfer bobot selesai!
>>> Final output shape tracking: (6,)


## Load Weights dari Model Conv2D (Layer-2-32-64-5x5-maxpool.h5)

Memuat bobot dari model Conv2D sebelumnya ke arsitektur LocallyConnected2D untuk memvalidasi forward pass secara ekuivalen.

## Evaluasi Keras

---

## From Scratch vs Keras (LC2D)

In [8]:
X_list, y_list = [], []
for X_batch, y_batch in test_ds:
    X_list.append(X_batch.numpy())
    y_list.append(y_batch.numpy())

X_test_lc = np.concatenate(X_list, axis=0)
y_true_lc = np.concatenate(y_list, axis=0).astype(int) 

print(f'X_test_lc : {X_test_lc.shape}')

X_test_lc : (3000, 150, 150, 3)


In [9]:
print('Pipeline from scratch:')
for i, layer in enumerate(model_scratch.layers):
    print(f'  [{i}] {type(layer).__name__}')
    if hasattr(layer, 'w'): 
        print(f'      -> weights shape: {layer.w.shape}')

probe = model_scratch.forward(X_test_lc[0])
assert probe.shape == (NUM_CLASSES,), f'Shape salah: {probe.shape}'
assert abs(probe.sum() - 1.0) < 1e-4, f'Tidak sum to 1: {probe.sum()}'
assert not np.any(np.isnan(probe)), 'NaN terdeteksi!'
print('\nSanity check OK')


Pipeline from scratch:
  [0] LocallyConnected2DLayer
  [1] MaxPooling2DLayer
  [2] LocallyConnected2DLayer
  [3] MaxPooling2DLayer
  [4] FlattenLayer
  [5] Dense
      -> weights shape: (73984, 128)
  [6] ActivationLayer
  [7] Dense
      -> weights shape: (128, 6)
  [8] ActivationLayer

Sanity check OK


In [10]:
# Forward pass from scratch — seluruh test set
print(f'Menjalankan from scratch pada {len(X_test_lc)} gambar...')
print('(LC2D lebih lambat dari Conv2D karena loop per-posisi tanpa vektorisasi)\n')

y_pred_scratch = np.zeros(len(X_test_lc), dtype=int)
t_start = time.perf_counter()

for i, img in enumerate(X_test_lc):
    y_pred_scratch[i] = model_scratch.predict(img)
    if (i + 1) % 500 == 0:
        elapsed = time.perf_counter() - t_start
        eta     = elapsed / (i + 1) * (len(X_test_lc) - i - 1)
        print(f'  {i+1}/{len(X_test_lc)} | {elapsed:.1f}s elapsed | ETA: {eta:.1f}s')

t_scratch  = time.perf_counter() - t_start
f1_scratch = f1_score(y_true_lc, y_pred_scratch, average='macro')

print(f'\nFrom Scratch F1: {f1_scratch:.4f}')
print(f'Waktu scratch  : {t_scratch:.2f} detik')


Menjalankan from scratch pada 3000 gambar...
(LC2D lebih lambat dari Conv2D karena loop per-posisi tanpa vektorisasi)

  500/3000 | 714.7s elapsed | ETA: 3573.3s
  1000/3000 | 1587.9s elapsed | ETA: 3175.8s
  1500/3000 | 2414.6s elapsed | ETA: 2414.6s
  2000/3000 | 3139.0s elapsed | ETA: 1569.5s
  2500/3000 | 3953.4s elapsed | ETA: 790.7s
  3000/3000 | 4637.1s elapsed | ETA: 0.0s

From Scratch F1: 0.7849
Waktu scratch  : 4637.07 detik
